# Bronze: Scraping Harga Shopee → DuckDB

Notebook ini **berdiri sendiri**, membaca daftar nama HP dari `data/gsmarena.duckdb`, dan menulis hasilnya ke
**`data/shopee.duckdb`** (tabel `listings`) — lapisan *bronze* untuk sumber Shopee.

Jika `data/tokopedia.duckdb` **sudah ada** (notebook Tokopedia sudah pernah dijalankan), keyword akan
**diprioritaskan** dari HP yang sudah punya listing di Tokopedia — supaya HP yang dicari di Shopee adalah HP
yang kemungkinan besar bisa **dipasangkan** untuk perbandingan harga nanti di `build_ducklake`. Ini murni
prioritas urutan, bukan ketergantungan wajib — notebook ini tetap bisa jalan sendiri tanpa `tokopedia.duckdb`.

## Kenapa Shopee berbeda dari Tokopedia: semi-manual
Dari pengalaman sebelumnya, Shopee **memblokir akses otomatis** ("Verifikasi gagal. Silakan coba lagi beberapa
menit lagi"). Karena itu notebook ini punya mode `SHOPEE_MODE = "manual"` (bawaan): **Anda yang mengetik
pencarian di Chrome**, notebook hanya **membaca** halaman hasil yang sedang tampil.

## Sebelum menjalankan
1. **`data/gsmarena.duckdb` harus sudah berisi data.**
2. Buka Chrome dengan remote debugging, buka `shopee.co.id`, dan **login**:
```
# Windows (Command Prompt)
"C:\Program Files\Google\Chrome\Application\chrome.exe" --remote-debugging-port=9222 --user-data-dir="C:\chrome_scrape"

# macOS
/Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome --remote-debugging-port=9222 --user-data-dir=/tmp/chrome_scrape

# Linux
google-chrome --remote-debugging-port=9222 --user-data-dir=/tmp/chrome_scrape
```
Pakai **satu tab Shopee saja** di jendela itu.

## 1. Instalasi & Import Library

In [ ]:
%pip install -q selenium pandas duckdb

In [1]:
import re
import os
import time
import random
from pathlib import Path
from datetime import datetime
from urllib.parse import quote_plus, urljoin

import pandas as pd
import duckdb
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 2. Konfigurasi

In [ ]:
DATA_DIR = Path("data")
DB_DIR = Path("database")
DATA_DIR.mkdir(exist_ok=True)
DB_DIR.mkdir(exist_ok=True)
GSMARENA_DB = DB_DIR / "gsmarena.duckdb"
TOKOPEDIA_DB = DB_DIR / "tokopedia.duckdb"    # opsional, hanya untuk prioritas urutan keyword
DB_PATH = DB_DIR / "shopee.duckdb"
CSV_PATH = DATA_DIR / "raw_dataset_shopee.csv"
print("Menulis ke:", DB_PATH.resolve())

GITHUB_WARN_MB = 50

DEBUG_PORT = 9222
# "manual" = Anda mengetik pencarian di Chrome, notebook hanya membaca hasil (PALING AMAN dari blokir, bawaan)
# "auto"   = notebook membuka halaman pencarian sendiri (rawan "Verifikasi gagal")
# "off"    = lewati Shopee
SHOPEE_MODE = "auto"

MAX_KEYWORDS = 300
MAX_PRODUCTS_PER_KEYWORD = 10
TARGET_ROWS = 600                 # Shopee cukup sebagai pelengkap (volume utama dari Tokopedia)
MAX_CONSECUTIVE_EMPTY = 12        # hanya dipakai di mode "auto"
DELAY = (8.0, 15.0)               # jauh lebih lambat dari Tokopedia — Shopee lebih sensitif

KEYWORD_EXCLUDE = ["pad", "tab", "fold", "flip", "watch"]
EXCLUDE_WORDS = ["case", "casing", "softcase", "tempered", "anti gores", "antigores", "charger",
                 "kabel", "cable", "holder", "sticker", "skin", "pouch", "sarung", "lcd", "sparepart",
                 "baterai", "battery", "earphone", "headset", "powerbank", "dummy", "replika", "mockup"]


def clean_price(text):
    if text is None:
        return None
    m = re.search(r"\d[\d\.,]*", str(text))
    if not m:
        return None
    digits = re.sub(r"[^\d]", "", m.group(0))
    return int(digits) if digits else None


def looks_like_accessory(product_name):
    low = (product_name or "").lower()
    return any(w in low for w in EXCLUDE_WORDS)


def save_csv():
    df = con.execute("SELECT * FROM listings ORDER BY keyword, scraped_at").df()
    try:
        df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    except PermissionError:
        alt = DB_DIR / f"{datetime.now():%Y%m%d_%H%M%S}_{CSV_PATH.name}"
        df.to_csv(alt, index=False, encoding="utf-8-sig")
        print(f"⚠️ File asli terkunci, disimpan sebagai: {alt}")
        return
    print(f"✔ {CSV_PATH} diperbarui ({len(df)} baris)")

Menulis ke: C:\KULIAH\Semester 7\Data Mining II\Tugas\PJBL\data\shopee.duckdb


## 3. Baca Daftar HP (diprioritaskan dari HP yang sudah ada di Tokopedia, jika tersedia)

In [4]:
if not GSMARENA_DB.exists():
    raise RuntimeError(f"{GSMARENA_DB} tidak ditemukan. Jalankan notebook 01_scraping_gsmarena.ipynb terlebih dahulu.")

gsm = duckdb.connect(str(GSMARENA_DB), read_only=True)
specs = gsm.execute("SELECT brand, phone_name, release_year FROM phone_specs").df()
gsm.close()

ok = ~specs["phone_name"].str.lower().str.split().apply(lambda ws: bool(set(ws) & set(KEYWORD_EXCLUDE)))
candidates = specs[ok].copy()

tokopedia_prioritized = False
if TOKOPEDIA_DB.exists():
    try:
        tk = duckdb.connect(str(TOKOPEDIA_DB), read_only=True)
        tk_counts = tk.execute("SELECT keyword, count(*) AS n FROM listings GROUP BY keyword").df()
        tk.close()
        candidates = candidates.merge(tk_counts, left_on="phone_name", right_on="keyword", how="left")
        candidates["n"] = candidates["n"].fillna(0)
        # HP dengan listing Tokopedia terbanyak diprioritaskan (kandidat pasangan terbaik), sisanya menyusul dari terbaru
        candidates = candidates.sort_values(["n", "release_year"], ascending=[False, False])
        keywords = candidates["phone_name"].head(MAX_KEYWORDS).tolist()
        print(f"Memprioritaskan keyword dari {TOKOPEDIA_DB.name} (ditemukan).")
        tokopedia_prioritized = True
    except duckdb.Error as e:
        print(f"⚠️ {TOKOPEDIA_DB.name} ada tetapi sedang terkunci (kemungkinan notebook Tokopedia masih "
              f"berjalan di kernel lain): {e}\n"
              "   Melanjutkan tanpa prioritas dari Tokopedia. Untuk prioritas lebih baik, restart kernel "
              "notebook Tokopedia (atau jalankan con.close() di situ), lalu jalankan ulang sel ini.")
if not tokopedia_prioritized:
    print(f"{TOKOPEDIA_DB.name} belum ada — keyword diurutkan dari HP terbaru saja (round-robin per brand).")
    groups = [g.sort_values("release_year", ascending=False, na_position="last")["phone_name"].tolist()
              for _, g in candidates.groupby("brand")]
    keywords = []
    for i in range(max((len(g) for g in groups), default=0)):
        for g in groups:
            if i < len(g):
                keywords.append(g[i])
    keywords = keywords[:MAX_KEYWORDS]

print(f"{len(keywords)} keyword siap dicari. Contoh: {keywords[:8]}")

Memprioritaskan keyword dari tokopedia.duckdb (ditemukan).
281 keyword siap dicari. Contoh: ['Apple iPhone Duo', 'Apple iPhone 18 Pro Max', 'Apple iPhone 17e', 'Huawei Mate XT 2 Ultimate', 'Huawei nova 16s Pro', 'Huawei Pura 90s Pro', 'Huawei MatePad Air (2026)', 'Huawei nova 16 Pro']


## 4. Koneksi ke `data/shopee.duckdb`

In [5]:
con = duckdb.connect(str(DB_PATH))
con.execute("""CREATE TABLE IF NOT EXISTS listings (
    keyword VARCHAR, product_name VARCHAR, product_url VARCHAR, price_discount_raw VARCHAR,
    price_original_raw VARCHAR, discount_label_raw VARCHAR, rating_raw VARCHAR, sold_raw VARCHAR,
    seller_name VARCHAR, seller_location VARCHAR, scraped_at TIMESTAMP
)""")
con.execute("""CREATE TABLE IF NOT EXISTS search_progress (
    keyword VARCHAR PRIMARY KEY, n_found INTEGER, searched_at TIMESTAMP
)""")

done_keywords = set(con.execute("SELECT keyword FROM search_progress").df()["keyword"])
n_rows = con.execute("SELECT count(*) FROM listings").fetchone()[0]
print(f"Checkpoint: {len(done_keywords)} keyword sudah dicari, {n_rows} listing tersimpan.")

Checkpoint: 0 keyword sudah dicari, 0 listing tersimpan.


## 5. Extract: Shopee (Selenium + BeautifulSoup)
Produk Shopee dikenali dari pola URL `-i.<id_toko>.<id_produk>` (stabil, tidak bergantung nama class CSS).
Halaman pencarian Shopee tidak menampilkan nama toko, jadi nama toko diisi `shop_<id>`.

In [6]:
BLOCK_PATTERNS = ["silakan coba lagi nanti", "verifikasi gagal", "coba lagi beberapa menit",
                  "too many requests", "access denied"]


def make_driver():
    opts = Options()
    opts.add_experimental_option("debuggerAddress", f"127.0.0.1:{DEBUG_PORT}")
    return webdriver.Chrome(options=opts)


def polite_sleep(a, b):
    time.sleep(random.uniform(a, b))


def scroll_page(driver, times=10):
    for _ in range(times):
        driver.execute_script("window.scrollBy(0, window.innerHeight * 0.8);")
        time.sleep(1.0)


def is_blocked(html):
    low = html.lower()
    return any(p in low for p in BLOCK_PATTERNS) and re.search(r"-i\.\d+\.\d+", html) is None


def needs_human(driver):
    url = driver.current_url.lower()
    if any(w in url for w in ["login", "verify", "captcha", "punish", "security"]):
        return True
    return is_blocked(driver.page_source)


def switch_to_shopee_tab(driver):
    for h in driver.window_handles:
        driver.switch_to.window(h)
        if "shopee" in driver.current_url.lower():
            return True
    return False


def extract_sold(tail):
    m = (re.search(r"terjual\s*([\d\.,]+\s*(?:rb|jt)?\+?)", tail, re.I)
         or re.search(r"([\d\.,]+\s*(?:rb|jt)?\+?)\s*terjual", tail, re.I))
    return m.group(1).strip() if m else None


def parse_shopee(html, keyword, limit):
    soup = BeautifulSoup(html, "lxml")
    rows, seen = [], set()
    for a in soup.find_all("a", href=True):
        m = re.search(r"-i\.(\d+)\.(\d+)", a["href"])
        if not m or m.group(2) in seen:
            continue
        nodes = list(a.stripped_strings)
        text = " ".join(nodes)
        price_matches = list(re.finditer(r"Rp\s?[\d\.]+", text))
        nums = sorted(p for p in (clean_price(x.group(0)) for x in price_matches) if p)
        if not nums:
            continue
        seen.add(m.group(2))

        idx = next((i for i, n in enumerate(nodes) if n.startswith("Rp")), len(nodes))
        pre = nodes[:idx]
        name = max(pre, key=len) if pre else text.split("Rp")[0].strip(" -|")

        tail = text[price_matches[-1].end():]
        disc = next((n for n in nodes if re.fullmatch(r"-?\d+\s*%", n)), None)
        loc = next((n for n in reversed(nodes)
                    if not re.search(r"\d|Rp|terjual", n, re.I) and 3 <= len(n) <= 30 and n != name), None)
        rating = re.search(r"\b([1-5][.,]\d)\b", tail)
        rows.append({
            "keyword": keyword, "product_name": name, "product_url": urljoin("https://shopee.co.id/", a["href"]).split("?")[0],
            "price_discount_raw": f"Rp{nums[0]}", "price_original_raw": f"Rp{nums[-1]}" if len(nums) > 1 else None,
            "discount_label_raw": disc, "rating_raw": rating.group(1) if rating else None,
            "sold_raw": extract_sold(tail), "seller_name": f"shop_{m.group(1)}", "seller_location": loc,
        })
        if len(rows) >= limit:
            break
    return rows

In [7]:
def scrape_shopee():
    if SHOPEE_MODE == "off":
        print("SHOPEE_MODE = 'off': dilewati.")
        return
    driver = make_driver()
    empty_streak = 0
    total = con.execute("SELECT count(*) FROM listings").fetchone()[0]

    for kw in keywords:
        if total >= TARGET_ROWS:
            print(f"✅ Target {TARGET_ROWS} baris tercapai.")
            break
        if kw in done_keywords:
            continue

        print(f"({total}/{TARGET_ROWS}) Keyword: {kw}")

        if SHOPEE_MODE == "manual":
            ans = input(f"   Ketik '{kw}' di kotak pencarian Shopee (Chrome). Setelah hasil tampil tekan Enter "
                        f"(s = lewati, q = berhenti): ").strip().lower()
            if ans == "q":
                print("Dihentikan. Jalankan ulang sel ini untuk melanjutkan.")
                break
            if ans == "s":
                con.execute("INSERT OR REPLACE INTO search_progress VALUES (?,0, now())", [kw])
                done_keywords.add(kw)
                continue
            if not switch_to_shopee_tab(driver):
                print("   Tab Shopee tidak ditemukan. Buka shopee.co.id, cari keyword tersebut, lalu coba lagi.")
                continue
        else:
            driver.get(f"https://shopee.co.id/search?keyword={quote_plus(kw)}")
            time.sleep(3)

        if is_blocked(driver.page_source):
            print("⛔ Shopee menampilkan halaman blokir/verifikasi. Proses dihentikan agar tidak memperburuk.\n"
                  "   Tunggu ± 15-30 menit / selesaikan verifikasi manual, lalu jalankan ulang sel ini.")
            break
        if needs_human(driver):
            input("   Login/captcha terdeteksi. Selesaikan di jendela Chrome, lalu tekan Enter di sini...")
            if SHOPEE_MODE != "manual":
                driver.get(f"https://shopee.co.id/search?keyword={quote_plus(kw)}")

        try:
            WebDriverWait(driver, 15).until(lambda d: len(d.find_elements(By.XPATH, "//a[contains(@href, '-i.')]")) > 0)
            scroll_page(driver)
            html = driver.page_source
        except TimeoutException:
            html = ""

        found = parse_shopee(html, kw, MAX_PRODUCTS_PER_KEYWORD) if html else []
        found = [r for r in found if not looks_like_accessory(r["product_name"])]
        print(f"   {len(found)} listing (setelah buang aksesoris)")

        if found:
            con.executemany(
                "INSERT INTO listings VALUES (?,?,?,?,?,?,?,?,?,?, now())",
                [[r["keyword"], r["product_name"], r["product_url"], r["price_discount_raw"], r["price_original_raw"],
                  r["discount_label_raw"], r["rating_raw"], r["sold_raw"], r["seller_name"], r["seller_location"]]
                 for r in found])
            total += len(found)
            empty_streak = 0
        else:
            empty_streak += 1

        con.execute("INSERT OR REPLACE INTO search_progress VALUES (?,?, now())", [kw, len(found)])
        done_keywords.add(kw)

        if SHOPEE_MODE != "manual" and empty_streak >= MAX_CONSECUTIVE_EMPTY:
            print(f"⛔ {MAX_CONSECUTIVE_EMPTY} keyword berturut-turut kosong. Jalankan diagnose() lalu ulangi.")
            break
        if SHOPEE_MODE != "manual":
            polite_sleep(*DELAY)

    save_csv()
    size_mb = DB_PATH.stat().st_size / 1_000_000
    flag = " ⚠️ mendekati/melebihi batas GitHub!" if size_mb > GITHUB_WARN_MB else ""
    print(f"\nUkuran {DB_PATH.name}: {size_mb:.1f} MB{flag}")


def diagnose(keyword="samsung galaxy a55"):
    driver = make_driver()
    driver.get(f"https://shopee.co.id/search?keyword={quote_plus(keyword)}")
    time.sleep(8)
    scroll_page(driver, 3)
    html = driver.page_source
    print("Judul halaman :", driver.title)
    print("URL sekarang  :", driver.current_url)
    print("Halaman diblokir?:", is_blocked(html))
    print("Butuh login/captcha?:", needs_human(driver))
    print("Hasil parser:", len(parse_shopee(html, keyword, 50)), "produk")
    driver.save_screenshot(str(DB_DIR / "debug_shopee.png"))
    (DB_DIR / "debug_shopee.html").write_text(html, encoding="utf-8")
    print(f"Screenshot & HTML disimpan di {DB_DIR}/debug_shopee.*")

## 6. Jalankan Scraping
Mode manual: ketik keyword yang diminta di Chrome, kembali ke notebook, tekan Enter.

In [19]:
scrape_shopee()

n = con.execute("SELECT count(*) FROM listings").fetchone()[0]
print(f"\nTotal listing tersimpan: {n}")
if n == 0 and SHOPEE_MODE != "off":
    print("⚠️ Belum ada listing. Jalankan diagnose() untuk memeriksa penyebabnya.")

✅ Target 600 baris tercapai.
✔ data\raw_dataset_shopee.csv diperbarui (606 baris)

Ukuran shopee.duckdb: 0.0 MB

Total listing tersimpan: 606


### Diagnosa (jalankan hanya jika hasil scraping kosong)

In [17]:
diagnose()

Judul halaman : Jual samsung galaxy a55 Harga Terbaik & Termurah Oktober 2026 | Shopee Indonesia
URL sekarang  : https://shopee.co.id/search?keyword=samsung+galaxy+a55
Halaman diblokir?: False
Butuh login/captcha?: False
Hasil parser: 3 produk
Screenshot & HTML disimpan di data/debug_shopee.*


## 7. Pratinjau Data

In [ ]:
con.execute("""
    SELECT keyword, count(*) AS jumlah_listing
    FROM listings GROUP BY keyword ORDER BY jumlah_listing DESC LIMIT 15
""").df()

In [ ]:
con.execute("SELECT * FROM listings ORDER BY scraped_at DESC LIMIT 10").df()

## Catatan
- **Lapisan bronze, belum dibersihkan**, sama seperti Tokopedia & GSMArena — pembersihan, pencocokan nama HP,
  dan penyamaan jumlah data dengan Tokopedia dilakukan di `build_ducklake`.
- **`data/raw_listings_shopee.csv`** hanya salinan untuk laporan.
- **Mode manual dipilih sebagai bawaan** karena Shopee memblokir akses otomatis jauh lebih agresif daripada
  Tokopedia. Ganti `SHOPEE_MODE = "auto"` hanya jika Anda yakin risikonya, dan `"off"` jika ingin mengandalkan
  Tokopedia saja untuk sementara.